<!-- gs-watermark -->
> **© 2026 Gokaran Shukla** · part of [orchid](https://github.com/Gokaran-Shukla/orchid) · MIT License — any reuse must keep this copyright notice and credit the author.

# ORCHID tutorial 10: Morphology, DFT couplings and MD trajectories

**Goals.** Go from molecules to hopping graphs: DIPRO transfer integrals (PySCF), dipolar site energies,
amorphous packings, and **MD trajectory import** (GROMACS `.gro`, multi-frame XYZ) with molecules made
whole across periodic boundaries (new in v0.4).

### Theory
DIPRO projects the dimer Fock matrix onto the monomer frontier orbitals: $J_{eff}=\frac{J-\frac12(e_A+e_B)S}{1-S^2}$.
For a symmetric dimer, $|J_{eff}|$ is half the orbital splitting. Transfer integrals decay as $J\propto e^{-\beta r}$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (5.4, 3.6), "axes.grid": True, "grid.alpha": 0.3})
import orchid
print("ORCHID", orchid.__version__)

ORCHID 0.4.0


In [2]:
from orchid.morphology import dipro_pyscf
eth = [("C", (0, 0, 0.667)), ("C", (0, 0, -0.667)), ("H", (0, 0.923, 1.232)),
       ("H", (0, -0.923, 1.232)), ("H", (0, 0.923, -1.232)), ("H", (0, -0.923, -1.232))]
for d in (3.6, 4.2, 5.0):
    r = dipro_pyscf(eth, [(s, (x + d, y, z)) for s, (x, y, z) in eth])
    print(f"d = {d} A: J_eff = {1e3*abs(r['J_eff']):.2f} meV, half splitting = {1e3*abs(r['splitting'])/2:.2f} meV")

d = 3.6 A: J_eff = 267.65 meV, half splitting = 268.15 meV
d = 4.2 A: J_eff = 66.89 meV, half splitting = 66.95 meV


d = 5.0 A: J_eff = 7.20 meV, half splitting = 7.21 meV


### Amorphous packing, dipolar energies, hopping graph

In [3]:
from orchid.morphology import random_packing, dipolar_site_energies, MolecularGraph
from orchid.master_equation import me_linear_response
box = np.array([14e-9] * 3)
pos = random_packing(700, box, 1.0e-9, seed=2)
E = dipolar_site_energies(pos, box, dipole_debye=4.0)
G = MolecularGraph(pos, box, r_cut=2.6e-9, energies=E)
print(f"{G.N} molecules, a = {G.a*1e9:.2f} nm, sigma = {1e3*E.std():.0f} meV")
print("mu(300 K) =", me_linear_response(G, 300.0, 1e-6, alpha=10.0)[0], "m^2/Vs")

700 molecules, a = 1.58 nm, sigma = 38 meV
mu(300 K) = 1.0922668443793609e-13 m^2/Vs


### MD trajectory import (v0.4)
`read_gro`/`read_xyz_trajectory` read the snapshots. `molecule_centres` unwraps every atom by the minimum
image relative to the first atom of its molecule, then takes the mass-weighted centre.
`trajectory_mobility` builds one graph per frame and averages the ME mobility over the frames.

In [4]:
import tempfile, os
from orchid.morphology import write_gro, read_gro, molecule_centres, trajectory_mobility
rng = np.random.default_rng(0)
ring = np.array([[np.cos(t), np.sin(t), 0] for t in np.arange(6) * np.pi / 3]) * 0.14e-9
box = np.array([10e-9] * 3)
centres = random_packing(500, box, 0.9e-9, seed=1)
frames = []
for f in range(3):                                   # three 'MD snapshots' with thermal jitter
    c = (centres + rng.normal(0, 0.05e-9, centres.shape)) % box
    xyz, atom, resid = [], [], []
    for i, ci in enumerate(c):
        Q, _ = np.linalg.qr(rng.normal(size=(3, 3)))
        xyz += [((ci + p) % box) for p in ring @ Q.T]; atom += ["C"] * 6; resid += [i + 1] * 6
    frames.append(dict(title="demo", resid=np.array(resid), resname=["BEN"] * len(atom), atom=atom,
                       xyz=np.array(xyz), box=box))
path = os.path.join(tempfile.mkdtemp(), "traj.gro")
write_gro(path, frames)
fr = read_gro(path)
cen, mol = molecule_centres(fr[0])
print(len(fr), "frames,", len(cen), "molecules")
r = trajectory_mobility(fr, 300.0, energies="gaussian", sigma=0.075, r_cut=2.0e-9)
print("mobility per frame:", r["mu_frames"], " mean", r["mu_mean"], "+-", r["mu_sem"])

3 frames, 500 molecules


mobility per frame: [4.36133608e-16 3.90042753e-15 1.57159632e-15]  mean 1.9693858181733963e-15 +- 1.0196421699788104e-15


Each snapshot is a separate static-disorder sample (new positions and new site energies), so in a
small box the per-frame mobilities scatter strongly. Production runs use larger boxes and more frames,
and report the frame average with its standard error, as here.

**Exercise.** Why would the naive average of atom positions fail for a molecule that straddles the box edge?

<details><summary>Solution</summary>

Atoms near $x=0$ and near $x=L$ would average to about $L/2$, far from either copy of the molecule. `molecule_centres` unwraps first, and V26 recovers 397 split molecules exactly.
</details>

## Quiz

Five questions on this section. Run the cell to see them, then grade yourself with
`quiz("10", answers="abcda")` (one letter per question). The same quiz runs in a terminal
with `orchid quiz 10`.

In [5]:
from orchid.tutorials import quiz
quiz("10")

Quiz 10: Morphology, DFT couplings and MD trajectories

Q1. For a symmetric dimer, DIPRO's effective transfer integral equals
    (a) Zero
    (b) Half the dimer orbital splitting
    (c) The full dimer splitting
    (d) The HOMO energy

Q2. Why must molecules be made whole across periodic boundaries before computing centres of mass?
    (a) GROMACS requires it
    (b) It changes the energies
    (c) To save memory
    (d) A molecule split by the boundary would otherwise get its centre somewhere in the middle of the box

Q3. Dipolar disorder produces site energies that are
    (a) Always zero on average with zero width
    (b) Uncorrelated
    (c) Spatially correlated, which extends the sqrt(F) Poole-Frenkel-like regime
    (d) Temperature dependent

Q4. The transfer integral between molecules typically decays with distance as
    (a) r^-1
    (b) r^-6
    (c) constant
    (d) exp(-beta r)

Q5. What does V26 prove about the graph solver?
    (a) MD is always needed
    (b) Mobility doe

In [6]:
# replace with your answers, e.g. "abcda"
# quiz("10", answers=".....")

<details><summary>Answer key</summary>

`bdcdc`. Run `quiz("10", answers="bdcdc")` to see the explanation for every question.
</details>

---
[Tutorial index](README.md) · ORCHID documentation · `orchid tutorials`